# 1. Libs

In [12]:
import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

# 2. Dataset

In [3]:
df = pd.read_csv("data/imdb_movies.csv")

print("Dimensões:", df.shape)

display(df.head())

Dimensões: (999, 16)


,Unnamed: 0,Series_Title,Released_Year,Certificate,Runtime,Genre,IMDB_Rating,Overview,Meta_score,Director,Star1,Star2,Star3,Star4,No_of_Votes,Gross
0,1,The Godfather,1972,A,175 min,"Crime, Drama",9.2,An organized crime dynasty's aging patriarch t...,100.0,Francis Ford Coppola,Marlon Brando,Al Pacino,James Caan,Diane Keaton,1620367,"134,966,411"
1,2,The Dark Knight,2008,UA,152 min,"Action, Crime, Drama",9.0,When the menace known as the Joker wreaks havo...,84.0,Christopher Nolan,Christian Bale,Heath Ledger,Aaron Eckhart,Michael Caine,2303232,"534,858,444"
2,3,The Godfather: Part II,1974,A,202 min,"Crime, Drama",9.0,The early life and career of Vito Corleone in ...,90.0,Francis Ford Coppola,Al Pacino,Robert De Niro,Robert Duvall,Diane Keaton,1129952,"57,300,000"
3,4,12 Angry Men,1957,U,96 min,"Crime, Drama",9.0,A jury holdout attempts to prevent a miscarria...,96.0,Sidney Lumet,Henry Fonda,Lee J. Cobb,Martin Balsam,John Fiedler,689845,"4,360,000"
4,5,The Lord of the Rings: The Return of the King,2003,U,201 min,"Action, Adventure, Drama",8.9,Gandalf and Aragorn lead the World of Men agai...,94.0,Peter Jackson,Elijah Wood,Viggo Mortensen,Ian McKellen,Orlando Bloom,1642758,"377,845,905"


In [ ]:
df.info()

In [ ]:
df.isnull().sum()

# 3. Columns selection

In [5]:
columns = [
    "Series_Title",
    "Released_Year",
    "Genre",
    "Director",
    "Star1",
    "Star2",
    "Star3",
    "Star4",
    "Overview",
    "IMDB_Rating"
]

movies = df[columns].copy()

# 4. Document Construction

In [6]:
def create_document(row):
    return (
        f"Title: {row['Series_Title']}\n"
        f"Year: {row['Released_Year']}\n"
        f"Genre: {row['Genre']}\n"
        f"Director: {row['Director']}\n"
        f"Star1: {row['Star1']}\n"
        f"Star2: {row['Star2']}\n"
        f"Star3: {row['Star3']}\n"
        f"Star4: {row['Star4']}\n"
        f"IMDb Rating: {row['IMDB_Rating']}\n"
        f"Overview: {row['Overview']}"
    )


movies["document"] = movies.apply(create_document, axis=1)

In [ ]:
print(movies["document"].iloc[0])

In [ ]:
print("Total documents:", len(movies))
print("Empty documents:", movies["document"].isna().sum())

# 5. Loading the embeding model

In [7]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

C:\Users\Rafael\movie-rag\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Rafael\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\Rafael\movie-rag\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarni

# 6. Embedding 

In [9]:
documents = movies["document"].tolist()

embeddings = model.encode(
    documents,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Shape dos embeddings:", embeddings.shape)

Batches: 100%|██████████| 32/32 [00:32<00:00,  1.01s/it]

Shape dos embeddings: (999, 384)


In [10]:
print(embeddings[0][:10])

[-0.10328615 -0.043273   -0.06441569 -0.01188476 -0.07674628  0.01037306
 -0.00513457 -0.01141561  0.02762846 -0.03084671]


# 7. VECTOR index

In [13]:
embeddings = np.asarray(embeddings, dtype="float32")

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("Total de vetores indexados:", index.ntotal)

Total de vetores indexados: 999


# 8. Seach function

In [14]:
def search_movies(query, top_k=5):

    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(query_embedding, top_k)

    results = movies.iloc[indices[0]].copy()
    results["similarity"] = scores[0]

    return results[
        ["Series_Title", "Genre", "Overview", "similarity"]
    ]

# 9. Testing

In [18]:
query = "Movies about organized crime families"

results = search_movies(query, top_k=20)

display(results)

,Series_Title,Genre,Overview,similarity
790,Road to Perdition,"Crime, Drama, Thriller","A mob enforcer's son witnesses a murder, forci...",0.500638
164,Casino,"Crime, Drama","A tale of greed, deception, money, power, and ...",0.498396
978,Lethal Weapon,"Action, Crime, Thriller",Two newly paired cops who are complete opposit...,0.484436
160,"Lock, Stock and Two Smoking Barrels","Action, Comedy, Crime",A botched card game in London triggers four fr...,0.484253
307,White Heat,"Action, Crime, Drama",A psychopathic criminal with a mother complex ...,0.483981
487,District 9,"Action, Sci-Fi, Thriller",Violence ensues after an extraterrestrial race...,0.482105
14,Goodfellas,"Biography, Crime, Drama",The story of Henry Hill and his life in the mo...,0.481330
950,Minority Report,"Action, Crime, Mystery",In a future where a special police unit is abl...,0.480426
5,Pulp Fiction,"Crime, Drama","The lives of two mob hitmen, a boxer, a gangst...",0.478042
0,The Godfather,"Crime, Drama",An organized crime dynasty's aging patriarch t...,0.472166
